In [ ]:
import sys
sys.path.append('../')

from src.run_app import *
from src.prepare_gam import *
from src.utils import *
from gam_rs_utils.binarize_dataset import binarize_dataset
from gam_rs_utils.utils import *
from FasterRisk.src.fasterrisk import fasterrisk
from time import time
import pickle

dataset_settings = {
    'bank': {
        'gap_tolerance': 0.0075,
        'num_estimators': 50,
    },
    'compas': {
        'gap_tolerance': 0.0025,
        'num_estimators': 50,
    },
    'diabetes': {
        'gap_tolerance': 0.0060,
        'num_estimators': 200,
    },
    # 'heloc_original'
    # 'hiv'
    'netherlands': {
        'gap_tolerance': 0.0020,
        'num_estimators': 50,
    },
    'spambase': {
        'gap_tolerance': 0.0055,
        'num_estimators': 50,
    },
}

results = []
for dataset_name, settings in dataset_settings.items():
    ne = settings["num_estimators"]
    gt = settings['gap_tolerance']

    dataset = pd.read_csv(path)
    print(f"Dataset: {dataset_name}")
    print(f"Binarized shape: {dataset.shape}")

    df, thresholds, header, threshold_guess_time = binarize_dataset(dataset, ne)
    X, y = df.iloc[:, :-1], df.iloc[:, -1]
    header = pd.Index(["intercept"] + list(X.columns)).astype("object")
    X_one_hot, y = utils.get_X_y(X, y)

    for fs in ['top', 'bottom'] + ['random'] * 5:
        start = time()
        rs = fasterrisk.RiskScoreOptimizer(X_one_hot, y, k=10, lb=-100, ub=100, gap_tolerance=gt, select_top_m=100, maxAttempts=5)
        rs.optimize_with_swaps(swaps=3, fanout_decay=1, feature_selection=fs)
        end = time()

        result = {
            "dataset": dataset_name,
            "dataset_shape": dataset.shape,
            "num_estimators": ne,
            "gap_tolerance": gt,
            "feature_selection": fs,
            "threshold_guess_time": threshold_guess_time,
            "num_features": len(header),
            "runtime": end - start,
            "betas": rs.sparseDiversePool_betas,
            "beta0": rs.sparseDiversePool_beta0,
            "num_solutions": rs.sparseDiversePool_betas.shape[0],
            "loss": get_loss(X_one_hot, y, rs.sparseDiversePool_beta0, rs.sparseDiversePool_betas)
        }
        results.append(result)

        print(f"\t{gt} tolerance, {result['num_solutions']} solutions, {result['runtime']:.2f} seconds, loss of {result['loss']:.4f}")

with open("results/order.pkl", "wb") as f:
    pickle.dump(results, f)

Dataset: bank
Binarized shape: (4521, 17)
	0.0075 tolerance, 100 solutions, 42.02 seconds, loss of 0.1044
	0.0075 tolerance, 0 solutions, 17.16 seconds, loss of 0.0000
	0.0075 tolerance, 100 solutions, 25.58 seconds, loss of 0.1045
	0.0075 tolerance, 100 solutions, 25.38 seconds, loss of 0.1045
	0.0075 tolerance, 100 solutions, 25.02 seconds, loss of 0.1046
	0.0075 tolerance, 100 solutions, 24.27 seconds, loss of 0.1048
	0.0075 tolerance, 100 solutions, 28.34 seconds, loss of 0.1045
Dataset: compas
Binarized shape: (6907, 8)
	0.0025 tolerance, 100 solutions, 46.44 seconds, loss of 0.3183
	0.0025 tolerance, 0 solutions, 12.27 seconds, loss of 0.0000
	0.0025 tolerance, 100 solutions, 33.62 seconds, loss of 0.3186
	0.0025 tolerance, 100 solutions, 30.69 seconds, loss of 0.3184
	0.0025 tolerance, 100 solutions, 36.65 seconds, loss of 0.3183
	0.0025 tolerance, 100 solutions, 36.05 seconds, loss of 0.3186
	0.0025 tolerance, 100 solutions, 31.56 seconds, loss of 0.3184
Dataset: diabetes
Binar